In [1]:
import sys
import os
import math
from typing import NamedTuple

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold

import jax
import jax.numpy as jnp
import optax


sys.path.append(os.path.join(os.getcwd(), 'import'))

from utils import read_utils
from utils import plot_utils
from utils import train_utils
from utils import artificializer

from models import clm
from models import hgm

In [2]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(artificializer)
importlib.reload(clm)
importlib.reload(hgm)

<module 'models.hgm' from '/home/julian-zim/Files/Cloud/OneDrive/OneFiles/Linux/Work/Workspaces/UNIVIE/JetBrains/DataSpell/MSc_Thesis/worm-2025-fieseler/import/models/hgm.py'>

# Functions

In [3]:
class Lambdas(NamedTuple):
	loss_lambdas: tuple[float, float]
	reg_lambdas: tuple[float, float]
	cca_lambdas: tuple[float, float]

class PrepData(NamedTuple):
	N_mean: tuple[float, ...]
	N_std: tuple[float, ...]
	B_hl_mean: tuple[float, ...] | None = None
	B_hl_std: tuple[float, ...] | None = None

In [4]:
def cosine_lr(step, warmup_steps, total_steps, lr_max, lr_min_ratio):
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    lr_min = lr_max * lr_min_ratio
    return lr_min + 0.5*(lr_max - lr_min)*(1 + math.cos(math.pi*progress))

In [5]:
def train_joint_model(model, splits, N, B_ll, B_hl, hyperparameters=None):
	# Hyperparamters ============
	seeds = jax.random.PRNGKey(0), jax.random.PRNGKey(1), jax.random.PRNGKey(2), jax.random.PRNGKey(3)

	if hyperparameters is None:
		hyperparameters = {
			'max_epochs': 1,
			'max_patience': 1,
			'learning_rate': 1E-3, #0.0001, #0.005, # 0.02
			'w1_reg_lambda': 0., #0.002,
			'w2_reg_lambda': 0., #0.5, #0.01,
			'ccadiag_lambda': 0.6,
			'ccaunit_lambda': 1 - 0.6,
			'loss1_lambda': 0.8,
			'loss2_lambda': 1. - 0.8,
			'dimensions': 2
		}

	max_epochs = hyperparameters['max_epochs']
	max_patience = hyperparameters['max_patience']

	learning_rate = hyperparameters['learning_rate']

	n_dimensions = hyperparameters['dimensions']

	lambdas = Lambdas(
		loss_lambdas=(hyperparameters['loss1_lambda'], hyperparameters['loss2_lambda']),
		reg_lambdas=(hyperparameters['w1_reg_lambda'], hyperparameters['w2_reg_lambda']),
		cca_lambdas=(hyperparameters['ccadiag_lambda'], hyperparameters['ccaunit_lambda'])
	)
	# ===========================

	n_neurons = N.shape[1]
	n_behaviors_hl = B_hl.shape[1]
	n_behaviors_ll = B_ll.shape[1]

	# training loop =======================================
	params_per_split = list()
	neuron_scores_per_split = list()
	losses_per_split = list()
	losses_tr_per_split = list()
	mse_losses_per_split = list()
	mse_losses_tr_per_split = list()
	__grads_per_split = list()
	for s_id, (train, test) in enumerate(splits):
		# splitting and initializing ========================== (this also moves the data to the GPU)
		N_train, N_test = jnp.array(N[train]), jnp.array(N[test])
		B_hl_train, B_hl_test = jnp.array(B_hl[train]), jnp.array(B_hl[test])
		B_ll_train, B_ll_test = jnp.array(B_ll[train]), jnp.array(B_ll[test])

		prep_data_train = PrepData(
			N_mean=tuple((map(float, N_train.mean(axis=0)))),
			N_std=tuple((map(float, jnp.where((N_std := jnp.std(N_train, axis=0)) == 0.0, 1.0, N_std)))),
			B_hl_mean=tuple((map(float, B_hl_train.mean(axis=0)))),
			B_hl_std=tuple((map(float, jnp.where((B_hl_std := jnp.std(B_hl_train, axis=0)) == 0.0, 1.0, B_hl_std))))
		)
		prep_data_test = PrepData(
			N_mean=tuple((map(float, N_test.mean(axis=0)))),
			N_std=tuple((map(float, jnp.where((N_std := jnp.std(N_test, axis=0)) == 0.0, 1.0, N_std)))),
			B_hl_mean=tuple((map(float, B_hl_test.mean(axis=0)))),
			B_hl_std=tuple((map(float, jnp.where((B_hl_std := jnp.std(B_hl_test, axis=0)) == 0.0, 1.0, B_hl_std))))
		)

		params = model.init_params(seeds,
								   (n_neurons, n_dimensions),
								   (n_behaviors_hl, n_dimensions),
								   (n_behaviors_ll, n_neurons),
								   * ((n_dimensions, n_neurons),) if model.get_type() else ())

		# optimizer = optax.adam(learning_rate=optax.cosine_decay_schedule(
		# 	init_value=learning_rate,
		# 	decay_steps=max_epochs,
		# 	alpha=1E-2
		# ))
		optimizer = optax.adam(learning_rate)
		opt_state = optimizer.init(params)
		# ===========================================================================================

		losses = [model.loss_fn(params, N_test, B_ll_test, B_hl_test, lambdas, prep_data_test).item()]
		losses_tr = [model.loss_fn(params, N_train, B_ll_train, B_hl_train, lambdas, prep_data_train).item()]
		mse_losses = [model.mse_loss_fn(params, N_test, B_ll_test, prep_data_test).item()]
		mse_losses_tr = [model.mse_loss_fn(params, N_train, B_ll_train, prep_data_train).item()]
		__grads = list()

		best_params = params
		best_loss = sys.float_info.max
		best_loss_tr = sys.float_info.max
		best_mse_loss = sys.float_info.max
		best_mse_loss_tr = sys.float_info.max

		# lr_step = 0
		patience = 0
		for epoch in range(max_epochs):
			params, opt_state, __grad = model.update(optimizer.update, opt_state, params, N_train, B_ll_train, B_hl_train, lambdas, prep_data_train)

			loss = model.loss_fn(params, N_test, B_ll_test, B_hl_test, lambdas, prep_data_test).item()
			loss_tr = model.loss_fn(params, N_train, B_ll_train, B_hl_train, lambdas, prep_data_train).item()
			mse_loss = model.mse_loss_fn(params, N_test, B_ll_test, prep_data_test).item()
			mse_loss_tr = model.mse_loss_fn(params, N_train, B_ll_train, prep_data_train).item()
			__grads.append(__grad)

			# early stopping ================
			if loss < best_loss:
				patience = 0

				best_params = params
				best_loss = loss
				best_loss_tr = loss_tr
				best_mse_loss = mse_loss
				best_mse_loss_tr = mse_loss_tr
			else:
				patience += 1
				if patience > max_patience:
					params = best_params
					losses = losses + [best_loss for _ in range(max_epochs - epoch)]
					losses_tr = losses_tr + [best_loss_tr for _ in range(max_epochs - epoch)]
					mse_losses = mse_losses + [best_mse_loss for _ in range(max_epochs - epoch)]
					mse_losses_tr = mse_losses_tr + [best_mse_loss_tr for _ in range(max_epochs - epoch)]

					print(f'\nStopped after {epoch} epochs.', end='')
					break

					# lr_step += 1
					# optimizer = optax.adam(learning_rate / (10 ** lr_step))
					# patience = 0
					# print(f'\nReduced learning rate from {learning_rate / (10 ** (lr_step - 1))} to {learning_rate / (10 ** lr_step)} after epoch {epoch} in split {s_id}', end='')

			losses.append(loss)
			losses_tr.append(loss_tr)
			mse_losses.append(mse_loss)
			mse_losses_tr.append(mse_loss_tr)
			# ===============================

		params_per_split.append(params)
		losses_per_split.append(losses)
		losses_tr_per_split.append(losses_tr)
		mse_losses_per_split.append(mse_losses)
		mse_losses_tr_per_split.append(mse_losses_tr)
		__grads_per_split.append(__grads)

		print(f'Split {s_id} completed. ', end='')

		# predicting and scoring ========================================================================
		score_per_neuron = model.score_fn(params, N_test, B_ll_test, prep_data_test)
		neuron_scores_per_split.append(score_per_neuron)
		# ===============================================================================================

	# =====================================================

	return params_per_split, np.array(neuron_scores_per_split), np.array(losses_per_split), np.array(losses_tr_per_split), np.array(mse_losses_per_split), np.array(mse_losses_tr_per_split), __grads_per_split

In [6]:
def train_joint_models(model, filenames, discrete_states, eigenworms, traces):
	models_params = list()
	models_scores = list()
	models_losses = list()
	models_losses_tr = list()
	models_mse_losses = list()
	models_mse_losses_tr = list()
	__models_grads = list()

	for i in range(len(filenames)):
		print(f'\n{i+1}/{len(filenames)}: {filenames[i]} ', end='')

		splits = list(StratifiedKFold(n_splits=5).split(np.zeros(traces[i].shape[0]), discrete_states[i]['REV'].values))

		(params_per_split,
		 neuron_scores_per_split,
		 losses_per_split, losses_tr_per_split,
		 mse_losses_per_split, mse_losses_tr_per_split,
		 __grads_per_split) = train_joint_model(model, splits,
												jnp.array(traces[i].values),
												jnp.array(eigenworms[i].values),
												jnp.array(discrete_states[i].values))

		models_params.append(params_per_split)
		models_scores.append(neuron_scores_per_split)
		models_losses.append(losses_per_split)
		models_losses_tr.append(losses_tr_per_split)
		models_mse_losses.append(mse_losses_per_split)
		models_mse_losses_tr.append(mse_losses_tr_per_split)
		__models_grads.append(__grads_per_split)

	return models_params, models_scores, models_losses, models_losses_tr, models_mse_losses, models_mse_losses_tr, __models_grads

In [7]:
def assemble(filenames, traces, clm_models_scores, hgm_models_scores):
	neurons_labels = [traces[i].columns for i in range(len(filenames))]

	neuron_scores = dict()
	clm_models_avg_scores = [np.mean(scores, axis=0) for scores in clm_models_scores]
	hgm_models_avg_scores = [np.mean(scores, axis=0) for scores in hgm_models_scores]
	for label_list, score_clm_list, score_hgm_list in zip(neurons_labels, clm_models_avg_scores, hgm_models_avg_scores):
		for i in range(len(label_list)):
			label = label_list[i]
			if label != '' and label[-1] != '_':
				score_clm_new, score_hgm_new = np.array(score_clm_list[i]), np.array(score_hgm_list[i])
				if len(score_hgm_new.shape) == 0:
					score_clm_new = score_clm_new.reshape(1, 1)
					score_hgm_new = score_hgm_new.reshape(1, 1)
				if label not in neuron_scores.keys():
					neuron_scores[label] = (score_clm_new, score_hgm_new)
				else:
					score_clm_prev, score_hgm_prev = neuron_scores[label]
					neuron_scores[label] = (np.vstack([score_clm_prev, score_clm_new]), np.vstack([score_hgm_prev, score_hgm_new]))

	return neuron_scores

In [8]:
def predict_wrapper(model, N, B_ll, params, neurons_labels):
	N_res_pred, N_res, N_rec = model.obtain(params,
											N, B_ll,
											PrepData(
												N_mean=tuple((map(float, N.mean(axis=0)))),
												N_std=tuple((map(float, jnp.where((N_std := jnp.std(N, axis=0)) == 0.0, 1.0, N_std)))),
											))
	return tuple(pd.DataFrame(data=traces, columns=neurons_labels) for traces in (N_res_pred, N_res, N_rec))

In [9]:
read_ids = (0, 33)
plot_id = 24 - read_ids[0]  # 33

# dims = 2
dims_generation = 2
weights = (0.5, 2.)

# selection_n = ['AVAR', 'AVEL', 'BAGR', 'BAGL', 'URYVR', 'URYVL']  # high m portion for worm ??
# selection_n = ['DB01', 'VB02', 'RMDDR', 'RMDDL', 'SMDDR', 'SMDVR']  # high c portion for worm 33
selection_n = ['VB02', 'SMDDR', 'SMDDL', 'RMDDR', 'RMDDL', 'SMDVR']  # high c portion for worm 24
selection_bhl = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']  # ['REV', 'VENTRAL_TURN']

# read & preprocess (GCamp & GFP)

In [10]:
path = os.path.join(read_utils.get_data_path(), 'worm-2025-fieseler', 'exported_data_')
filenames_gcamp, continuous_behaviors_gcamp, discrete_states_gcamp, eigenworms_gcamp, kymographs_gcamp, traces_gcamp = read_utils.get_data(path, slice=(read_ids[0], read_ids[1] + 1))

for i in range(len(filenames_gcamp)):
	mask = eigenworms_gcamp[i].notna().all(axis=1)
	#continuous_behaviors_gcamp[i] = continuous_behaviors_gcamp[i][mask]
	discrete_states_gcamp[i] = discrete_states_gcamp[i][mask]
	eigenworms_gcamp[i] = eigenworms_gcamp[i][mask]
	traces_gcamp[i] = traces_gcamp[i][mask]

	#kymographs_gcamp[i] = kymographs_gcamp[i][mask]
	#filenames_gcamp[i] = filenames_gcamp[i][mask]

discrete_states_gcamp = [ds[selection_bhl] for ds in discrete_states_gcamp]


path_gfp = os.path.join(read_utils.get_data_path(), 'worm-2025-fieseler', 'exported_data_gfp')
filenames_gfp, continuous_behaviors_gfp, discrete_states_gfp, eigenworms_gfp, kymographs_gfp, traces_gfp = read_utils.get_data(path_gfp, slice=(0, 7+1))

for i in range(len(filenames_gfp)):
	mask = eigenworms_gfp[i].notna().all(axis=1)
	#continuous_behaviors_gfp[i] = continuous_behaviors_gfp[i][mask]
	discrete_states_gfp[i] = discrete_states_gfp[i][mask]
	eigenworms_gfp[i] = eigenworms_gfp[i][mask]
	traces_gfp[i] = traces_gfp[i][mask]

	#kymographs_gfp[i] = kymographs_gfp[i][mask]
	#filenames_gfp[i] = filenames_gfp[i][mask]

discrete_states_gfp = [ds[selection_bhl] for ds in discrete_states_gfp]

# ARTIFICIAL DATA

In [11]:
filenames_art = list()
discrete_states_art = discrete_states_gcamp.copy()
eigenworms_art = eigenworms_gcamp.copy()
neural_modes_art = list()
traces_reconstructed_art = list()
traces_residual_art = list()
traces_art = list()
for i in range(len(filenames_gcamp)):
	(filename_art,
	 neural_modes_art_,
	 traces_reconstructed_art_,
	 traces_residual_art_,
	 traces_art_,
	 params_art_,
	 params_cca_art_) = artificializer.artificialize(weights, hgm,
													 filenames_gcamp[i],
													 traces_gcamp[i].columns,
													 eigenworms_art[i],
													 discrete_states_art[i],
													 n_dimensions=dims_generation)
	filenames_art.append(filename_art)
	neural_modes_art.append(neural_modes_art_)
	traces_reconstructed_art.append(traces_reconstructed_art_)
	traces_residual_art.append(traces_residual_art_)
	traces_art.append(traces_art_)

In [12]:
# plot_utils.plot_traces(f'[{filenames_art[plot_id]}] traces residuals artificial',
# 					   discrete_states_art[plot_id]['REV'],
# 					   traces_residual_art[plot_id][selection_n],
# 					   n_plots=len(selection_n), plots_per_row=2, ylimits=None)

In [ ]:
# train clm art

(clm_models_params_art,
 clm_models_scores_art,
 clm_models_losses_art, clm_models_losses_tr_art,
 clm_models_mse_losses_art, clm_models_mse_losses_tr_art,
 __clm_models_grads_art) = train_joint_models(clm,
											  filenames_art,
											  discrete_states_art,
											  eigenworms_art,
											  traces_art)


1/34: ZIMMERMANN_artificial_worm3-2022_11_28_no_image_data  Split 0 completed. Split 1 completed. Split 2 completed. Split 3 completed. Split 4 completed. 
2/34: ZIMMERMANN_artificial_worm3-2022-12-10_no_image_data  Split 0 completed. Split 1 completed. Split 2 completed. Split 3 completed. Split 4 completed. 
3/34: ZIMMERMANN_artificial_worm6-2022-12-10_no_image_data  Split 0 completed. Split 1 completed. Split 2 completed. Split 3 completed. Split 4 completed. 
4/34: ZIMMERMANN_artificial_worm7-2022-12-10_no_image_data  

In [ ]:
# train hgm art

(hgm_models_params_art,
 hgm_models_scores_art,
 hgm_models_losses_art, hgm_models_losses_tr_art,
 hgm_models_mse_losses_art, hgm_models_mse_losses_tr_art,
 __hgm_models_grads_art) = train_joint_models(hgm,
											  filenames_art,
											  discrete_states_art,
											  eigenworms_art,
											  traces_art)

In [ ]:
plot_utils.plot_losses_splits('artificial hgm vs clm: train',
							  filenames_art,
							  hgm_models_losses_tr_art, clm_models_losses_tr_art,
							  plots_per_row=4, ylimits=None)

In [ ]:
plot_utils.plot_losses_splits('artificial hgm vs clm: test',
							  filenames_art,
							  hgm_models_losses_art, clm_models_losses_art,
							  plots_per_row=4, ylimits=None)

### traces check

In [ ]:
clm_pres_res_rec_art = [[
	predict_wrapper(clm, jnp.array(traces_art[i]), jnp.array(eigenworms_art[i]), params, traces_art[i].columns) for params in clm_models_params_art[i]
] for i in range(len(traces_art))]

hgm_pres_res_rec_art = [[
	predict_wrapper(hgm, jnp.array(traces_art[i]), jnp.array(eigenworms_art[i]), params, traces_art[i].columns) for params in hgm_models_params_art[i]
] for i in range(len(traces_art))]

In [ ]:
plot_utils.plot_traces_splits(f'worm {plot_id} hgm definition manifold',
							  discrete_states_art[plot_id]['REV'],
							  traces_art[plot_id][selection_n], [split[2] for split in hgm_pres_res_rec_art[plot_id]],
							  plots_per_row=2, ylimits=None)
plot_utils.plot_traces_splits(f'worm {plot_id} hgm definition residuals',
							  discrete_states_art[plot_id]['REV'],
							  traces_residual_art[plot_id][selection_n], [split[1] for split in hgm_pres_res_rec_art[plot_id]],
							  plots_per_row=2, ylimits=None)
plot_utils.plot_traces_splits(f'worm {plot_id} hgm residuals prediction',
							  discrete_states_art[plot_id]['REV'],
							  traces_residual_art[plot_id][selection_n], [split[0] for split in hgm_pres_res_rec_art[plot_id]],
							  plots_per_row=2, ylimits=None)

# plot_utils.plot_traces_splits(f'worm {plot_id} clm: definition manifold',
# 							  discrete_states_art[plot_id]['REV'],
# 							  traces_art[plot_id][selection_n], [split[2] for split in clm_pres_res_rec_art[plot_id]],
# 							  plots_per_row=2, ylimits=None)
# plot_utils.plot_traces_splits(f'worm {plot_id} clm: definition residuals',
# 							  discrete_states_art[plot_id]['REV'],
# 							  traces_residual_art[plot_id][selection_n], [split[1] for split in clm_pres_res_rec_art[plot_id]],
# 							  plots_per_row=2, ylimits=None)
plot_utils.plot_traces_splits(f'worm {plot_id} clm: residuals prediction',
							  discrete_states_art[plot_id]['REV'],
							  traces_residual_art[plot_id][selection_n], [split[0] for split in clm_pres_res_rec_art[plot_id]],
							  plots_per_row=2, ylimits=None)

In [ ]:
# def predict_wrapper2(model, N, B_ll, params):
# 	N_res_pred, N_res, N_rec, (N_man, linterm, sigterm) = model.obtain2(params,
# 											N, B_ll,
# 											PrepData(
# 												N_mean=tuple((map(float, N.mean(axis=0)))),
# 												N_std=tuple((map(float, jnp.where((N_std := jnp.std(N, axis=0)) == 0.0, 1.0, N_std)))),
# 											))
# 	return np.array(N_res_pred), np.array(N_res), np.array(N_rec), (np.array(N_man), np.array(linterm), np.array(sigterm))
#
#
# split_id = 2
#
# hgm_stuff = [predict_wrapper2(hgm, np.array(traces_art[0]), np.array(eigenworms_art[0]), params) for params in hgm_models_params_art[0]]
# TEMP_N_res_pred, TEMP_N_res, TEMP_N_rec, (TEMP_N_man, TEMP_linterm, TEMP_sigterm) = hgm_stuff[split_id]
#
#
# TEMP_N_res_pred = pd.DataFrame(data=TEMP_N_res_pred, columns=traces_art[0].columns)
# TEMP_N_res = pd.DataFrame(data=TEMP_N_res, columns=traces_art[0].columns)
# TEMP_N_rec = pd.DataFrame(data=TEMP_N_rec, columns=traces_art[0].columns)
# TEMP_N_man = pd.DataFrame(data=TEMP_N_man, columns=['CC1'])
# TEMP_linterm = pd.DataFrame(data=TEMP_linterm, columns=traces_art[0].columns)
# TEMP_sigterm = pd.DataFrame(data=TEMP_sigterm, columns=traces_art[0].columns)
#
#
# p2 = pd.DataFrame(data=TEMP_linterm.values * TEMP_sigterm.values, columns=traces_residual_art[0].columns)
# plot_utils.plot_traces('CC1',
# 					   discrete_states_art[plot_id]['REV'],
# 					   [TEMP_N_man],
# 					   keys=['CC1'],
# 					   n_plots=1, ylimits=(-0.5, 0.5))
#
# plot_utils.plot_traces('AVAR',
# 					   discrete_states_art[plot_id]['REV'],
# 					   [TEMP_sigterm],
# 					   keys=['AVAR'],
# 					   n_plots=1, ylimits=(0.49, 0.51))

### performance

In [ ]:
# assemble_art

neuron_scores_art = assemble(filenames_art, traces_art, clm_models_scores_art, hgm_models_scores_art)

In [ ]:
plot_utils.plot_performance_plot(neuron_scores_art, None,
								 limits=None, content_size=20)

In [ ]:
plot_utils.plot_improvement_plot(neuron_scores_art, None,
								 limits=None, content_size=20)

# ARTIFICIAL DATA END

In [ ]:
if False: raise Exception

# GCAMP

### CLM

In [ ]:
(clm_models_params_gcamp,
 clm_models_scores_gcamp,
 clm_models_losses_gcamp, clm_models_losses_tr_gcamp,
 clm_models_mse_losses_gcamp, clm_models_mse_losses_tr_gcamp,
 __clm_models_grads_gcamp) = train_joint_models(clm,
												filenames_gcamp,
												discrete_states_gcamp,
												eigenworms_gcamp,
												traces_gcamp)

### HGM

In [ ]:
(hgm_models_params_gcamp,
 hgm_models_scores_gcamp,
 hgm_models_losses_gcamp, hgm_models_losses_tr_gcamp,
 hgm_models_mse_losses_gcamp, hgm_models_mse_losses_tr_gcamp,
 __hgm_models_grads_gcamp) = train_joint_models(hgm,
												filenames_gcamp,
												discrete_states_gcamp,
												eigenworms_gcamp,
												traces_gcamp)

In [ ]:
plot_utils.plot_losses_splits('gcamp hgm vs clm: train',
							  filenames_gcamp,
							  hgm_models_losses_tr_gcamp, clm_models_losses_tr_gcamp,
							  plots_per_row=4, ylimits=None)

In [ ]:
plot_utils.plot_losses_splits('gcamp hgm vs clm: test',
							  filenames_gcamp,
							  hgm_models_losses_gcamp, clm_models_losses_gcamp,
							  plots_per_row=4, ylimits=None)

### traces check

In [ ]:
clm_pres_res_rec_gcamp = [[
	predict_wrapper(clm, jnp.array(traces_gcamp[i]), jnp.array(eigenworms_gcamp[i]), params, traces_gcamp[i].columns) for params in clm_models_params_gcamp[i]
] for i in range(len(traces_gcamp))]

hgm_pres_res_rec_gcamp = [[
	predict_wrapper(hgm, jnp.array(traces_gcamp[i]), jnp.array(eigenworms_gcamp[i]), params, traces_gcamp[i].columns) for params in hgm_models_params_gcamp[i]
] for i in range(len(traces_gcamp))]

In [ ]:
plot_utils.plot_traces_splits(f'worm {plot_id} hgm definition manifold',
							  discrete_states_gcamp[plot_id]['REV'],
							  traces_gcamp[plot_id][selection_n], [split[2] for split in hgm_pres_res_rec_gcamp[plot_id]],
							  plots_per_row=2, ylimits=(-5, 5))
plot_utils.plot_traces_splits(f'worm {plot_id} hgm definition residuals',
							  discrete_states_gcamp[plot_id]['REV'],
							  traces_gcamp[plot_id][selection_n], [split[1] for split in hgm_pres_res_rec_gcamp[plot_id]],
							  plots_per_row=2, ylimits=(-5, 5))
plot_utils.plot_traces_splits(f'worm {plot_id} hgm residuals prediction',
							  discrete_states_gcamp[plot_id]['REV'],
							  traces_gcamp[plot_id][selection_n], [split[0] for split in hgm_pres_res_rec_gcamp[plot_id]],
							  plots_per_row=2, ylimits=(-5, 5))

# plot_utils.plot_traces_splits(f'worm {plot_id} clm: definition manifold',
# 							  discrete_states_gcamp[plot_id]['REV'],
# 							  traces_gcamp[plot_id][selection_n], [split[2] for split in clm_pres_res_rec_gcamp[plot_id]],
# 							  plots_per_row=2, ylimits=(-5, 5))
# plot_utils.plot_traces_splits(f'worm {plot_id} clm: definition residuals',
# 							  discrete_states_gcamp[plot_id]['REV'],
# 							  traces_gcamp[plot_id][selection_n], [split[1] for split in clm_pres_res_rec_gcamp[plot_id]],
# 							  plots_per_row=2, ylimits=(-5, 5))
plot_utils.plot_traces_splits(f'worm {plot_id} clm: residuals prediction',
							  discrete_states_gcamp[plot_id]['REV'],
							  traces_gcamp[plot_id][selection_n], [split[0] for split in clm_pres_res_rec_gcamp[plot_id]],
							  plots_per_row=2, ylimits=(-5, 5))

### performance

In [ ]:
neuron_scores_gcamp = assemble(filenames_gcamp, traces_gcamp, clm_models_scores_gcamp, hgm_models_scores_gcamp)

In [ ]:
plot_utils.plot_performance_plot(neuron_scores_gcamp, None,
								 limits=None, content_size=20)

In [ ]:
plot_utils.plot_improvement_plot(neuron_scores_gcamp, None,
								 limits=None, content_size=20)

### __loss gradient per weights over time

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(12, 3))
# for ax in axes:
#     ax.set_ylim(-0.005, 0.004)
#
# for i in range(1):
# 	gradsW = np.array([val['W'] for val in __clm_models_grads_gcamp[-1][i]])
#
# 	gradsW1 = np.array([val['W1'] for val in __hgm_models_grads_gcamp[-1][i]])
# 	gradsW2 = np.array([val['W2'] for val in __hgm_models_grads_gcamp[-1][i]])
#
#
# 	axes[0].plot(np.mean(gradsW1, axis=1))
#
# 	# axes[1].plot(np.mean(gradsW1, axis=1))
# 	axes[1].plot(np.mean(gradsW2, axis=1))

In [ ]:
if False: raise Exception

# GFP

### CLM

In [ ]:
(clm_models_params_gfp,
 clm_models_scores_gfp,
 clm_models_losses_gfp, clm_models_losses_tr_gfp,
 clm_models_mse_losses_gfp, clm_models_mse_losses_tr,
 __clm_models_grads_gfp) = train_joint_models(clm,
											  filenames_gfp,
											  discrete_states_gfp,
											  eigenworms_gfp,
											  traces_gfp)

### HGM

In [ ]:
(hgm_models_params_gfp,
 hgm_models_scores_gfp,
 hgm_models_losses_gfp, hgm_models_losses_tr_gfp,
 hgm_models_mse_losses_gfp, hgm_models_mse_losses_tr_gfp,
 __hgm_models_grads_gfp) = train_joint_models(hgm,
											  filenames_gfp,
											  discrete_states_gfp,
											  eigenworms_gfp,
											  traces_gfp)

In [ ]:
plot_utils.plot_losses_splits('gfp hgm vs clm: train',
							  filenames_gfp,
							  hgm_models_losses_tr_gfp, clm_models_losses_tr_gfp,
							  plots_per_row=4, ylimits=None)

In [ ]:
plot_utils.plot_losses_splits('gfp hgm vs clm: test',
							  filenames_gfp,
							  hgm_models_losses_gfp, clm_models_losses_gfp,
							  plots_per_row=4, ylimits=None)

# assemble

In [ ]:
neuron_scores_gfp = assemble(filenames_gfp, traces_gfp, clm_models_scores_gfp, hgm_models_scores_gfp)

# Plot

In [ ]:
plot_utils.plot_performance_plot(neuron_scores_gcamp, neuron_scores_gfp,
								 limits=None, content_size=20)

In [ ]:
plot_utils.plot_improvement_plot(neuron_scores_gcamp, neuron_scores_gfp,
								 limits=None, content_size=20)